# Zero-Train Optimization of a Healthcare Chatbot
Medical Chatbot:

Background: Medical advice chatbot (LLM) classifier model trained on based on Llama3.2 on online medical forums.


Dataset: https://huggingface.co/datasets/lextale/FirstAidInstructionsDataset/viewer/default/icliniqDataset


Scenario: Optimize the model without additional training.


Use Case: The model is already trained and we want to optimize it without additional training.

## Setup

### Install dependencies and utility functions: This cell should be run once.

In [1]:
%%capture

%pip install matplotlib numpy pandas requests tqdm ipywidgets
%pip install --ignore-installed 'git+https://github.com/Authentrics-ai/authentrics-client.git@v2.1.0'

from pathlib import Path

import os
import authentrics_client as authrx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import random
from datetime import datetime
import time
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from threading import Thread
import contextlib

# Utility functions for loading indicators
@contextlib.contextmanager
def loading_spinner(message="Processing..."):
    """Context manager for showing a loading spinner"""
    spinner_widget = widgets.HTML(value=f"""
    <div style="display: flex; align-items: center; gap: 10px;">
        <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
        <span style="font-size: 14px; color: #555;">{message}</span>
    </div>
    <style>
    @keyframes spin {{
        0% {{ transform: rotate(0deg); }}
        100% {{ transform: rotate(360deg); }}
    }}
    </style>
    """)
    display(spinner_widget)
    try:
        yield
    finally:
        spinner_widget.close()

def show_progress_bar(total, description="Progress"):
    """Create and return a progress bar"""
    return tqdm(total=total, desc=description, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

def wait_for_analytics(project_id, min_wait_time=30, max_wait_time=300, check_interval=10):
    """
    Wait for ALL analytics to be complete before continuing.
    This prevents NaN values from appearing in the summary table.
    """
    import time

    print("🔄 Waiting for analytics to process...")
    countdown_widget = widgets.HTML()
    display(countdown_widget)

    # Minimum wait with countdown
    for remaining in range(min_wait_time, 0, -1):
        countdown_widget.value = f"""
        <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
            <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
            <span style="font-size: 14px; color: #555;">Minimum wait period: {remaining} seconds remaining...</span>
        </div>
        <style>
        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}
        </style>
        """
        time.sleep(1)

    # Poll until ALL analytics are complete
    start_polling = time.time()
    attempt = 1

    while (time.time() - start_polling) < (max_wait_time - min_wait_time):
        try:
            project = client.project.get_project_by_id(project_id)
            files = project["fileList"][1:]  # Skip base model

            # Check that ALL files have both weight and bias contributions
            files_missing_analytics = []
            for f in files:
                if (f.get("totalWeightContribution") is None or
                    f.get("totalBiasContribution") is None):
                    files_missing_analytics.append(f["fileName"])

            # Only continue if ALL analytics are complete
            if not files_missing_analytics:
                countdown_widget.value = """
                <div style="color: green; font-weight: bold; margin: 10px 0;">
                    ✅ All analytics are complete!
                </div>
                """
                return True

            # Show progress with specific missing files
            total_files = len(files)
            completed_files = total_files - len(files_missing_analytics)
            progress_pct = (completed_files / total_files * 100) if total_files > 0 else 0

            missing_display = ', '.join(files_missing_analytics[:3])
            if len(files_missing_analytics) > 3:
                missing_display += f" and {len(files_missing_analytics) - 3} more"

            countdown_widget.value = f"""
            <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
                <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #orange; border-radius: 50%; animation: spin 1s linear infinite;"></div>
                <span style="font-size: 14px; color: #555;">
                    Attempt {attempt}: {completed_files}/{total_files} files complete ({progress_pct:.1f}%)
                    <br/>⏳ Waiting for: {missing_display}
                </span>
            </div>
            <style>
            @keyframes spin {{
                0% {{ transform: rotate(0deg); }}
                100% {{ transform: rotate(360deg); }}
            }}
            </style>
            """
            attempt += 1
            time.sleep(check_interval)

        except Exception as e:
            print(f"⚠️ Error checking analytics status: {e}")
            time.sleep(check_interval)

    # Timeout reached - show final status
    try:
        project = client.project.get_project_by_id(project_id)
        files = project["fileList"][1:]
        files_missing_analytics = [f["fileName"] for f in files
                                 if (f.get("totalWeightContribution") is None or
                                     f.get("totalBiasContribution") is None)]

        if files_missing_analytics:
            countdown_widget.value = f"""
            <div style="color: red; font-weight: bold; margin: 10px 0;">
                ⚠️ Timeout: Analytics incomplete for {len(files_missing_analytics)} files.
                <br/>NaN values will appear for: {', '.join(files_missing_analytics[:5])}
                <br/>Consider increasing wait time or contacting support.
            </div>
            """
            return False
        else:
            countdown_widget.value = """
            <div style="color: green; font-weight: bold; margin: 10px 0;">
                ✅ All analytics completed just in time!
            </div>
            """
            return True
    except Exception as e:
        countdown_widget.value = f"""
        <div style="color: red; font-weight: bold; margin: 10px 0;">
            ❌ Error checking final status: {e}
        </div>
        """
        return False

def inference(optimized_model_path, prompt):
    base_model_path = "gs://authentrics-prod-bucket/demo-models/Medical-Chatbot/iteration_24.tar"
    response = requests.post(
        DEMO_SERVER_URL + "/inference",
        json={
            "prompt": prompt,
            "models": [base_model_path, optimized_model_path],
        },
    )
    response.raise_for_status()
    response = response.json()
    print(f"User prompt:\n\n\t{response['prompt']}\n")
    print(
        "Base model response:"
        f"\n\n\t{response['model_responses'][0]}\n"
    )
    print(
        "Optimized model response:"
        f"\n\n\t{response['model_responses'][1]}\n"
    )

def display_static_analysis_results(result: dict):
    print(f"Weight summary score: {result['weight_summary_score']}")
    pd.options.display.max_rows = None

        
    methods = {"min": np.amin, "mean": np.mean, "std": np.std, "max": np.amax}
    awd = {"name": [], "min": [], "mean": [], "std": [], "max": []}

    for name, values in response["result"]["absolute_weight_difference"].items():
        awd["name"].append(name)
        for method_name, method in methods.items():
            awd[method_name].append(method(values))

    df = pd.DataFrame(awd)
    display(df)


### Contact Authentrics for the URL and user credentials info@authentrics.ai

In [2]:

DEMO_SERVER_URL = input("Enter your Authentrics URL: ")
PROJECT_NAME = "Healthcare Chatbot ZTO"
pd.options.display.precision = 3
pd.options.display.chop_threshold = None

### Establish a session with authentrics
- Rerun this cell if your session expires

In [3]:
client = authrx.AuthentricsClient(DEMO_SERVER_URL)
client.auth.login()
print("✅ Session established successfully!")

✅ Session established successfully!


### Check that the inference server is running

In [4]:
requests.get(DEMO_SERVER_URL + "/health", timeout=5).raise_for_status()
print("✅ Inference server is running!")

✅ Inference server is running!


### Generate a project in Authentrics to track checkpoints, and perform asynchronous analytics
1. Creates a project for checkpoint management.
2. **Option A**: Point to existing checkpoint storage (no data movement required).
3. **Option B**: Upload checkpoints directly through our client (automated storage and versioning).

In [5]:
projects = client.project.get_projects()
if projects is not None:
    project = next((p for p in projects if p["name"].startswith(PROJECT_NAME)), None)
    if project:
        client.project.delete_project(project["id"], hard_delete=True)

PROJECT_NAME = PROJECT_NAME + " " + str(int(datetime.now().timestamp()))
project = client.project.create_project(
    PROJECT_NAME,
    "A smaller LLM specializing in medical advice",
    authrx.FileType.HF_TEXT,
)
project_id = project["id"]

### Restore all checkpoint files, stimulus files, and expected output file in bucket to a stable version

In [6]:
with loading_spinner("Please wait while we restore the requested files..."):
    client.get('/transfer/MedChat')

HTML(value='\n    <div style="display: flex; align-items: center; gap: 10px;">\n        <div style="width: 20p…

### Point authentrics file registry to the checkpoints using the api

In [7]:
print("Adding checkpoints to project...")
with show_progress_bar(5, "Adding checkpoints") as pbar:
    for i in range(20, 25):
        client.checkpoint.add_external_checkpoint(
            project_id,
            f"demo-models/Medical-Chatbot/iteration_{i}.tar",
            authrx.FileType.HF_TEXT,
            file_name=f"iteration_from_dataset_{i}.tar",
            tag=f"v{i}",
        )
        pbar.update(1)

project = client.project.get_project_by_name(PROJECT_NAME)
project_id = project["id"]
print(f"✅ Project created with id: {project['id']}")
print(f"Project name: {project['name']}")
for file in project["fileList"]:
    print(f"File name: {file['fileName']}")

Adding checkpoints to project...


Adding checkpoints:   0%|          | 0/5 [00:00<?]

✅ Project created with id: 693325a950eb215a61201da7
Project name: Healthcare Chatbot ZTO 1764959657
File name: iteration_from_dataset_20.tar
File name: iteration_from_dataset_21.tar
File name: iteration_from_dataset_22.tar
File name: iteration_from_dataset_23.tar
File name: iteration_from_dataset_24.tar


Wait for Analytics to be ready before proceeding.

In [8]:
print("\n" + "="*60)
print("📊 ANALYTICS PROCESSING")
print("="*60)
print("Authentrics is now computing analytics in the background.")
print("This includes weight contributions and bias scores for each checkpoint.")

analytics_ready = wait_for_analytics(project_id, min_wait_time=10, max_wait_time=300)
if not analytics_ready:
    print("\n❌ Analytics are not complete.")
    print("Please wait a moment longer and then continue. Please contact support if the issue persists.")


📊 ANALYTICS PROCESSING
Authentrics is now computing analytics in the background.
This includes weight contributions and bias scores for each checkpoint.
🔄 Waiting for analytics to process...


HTML(value='')

## Zero-Train Optimization (ZTO) Results

Authentrics.ai provides a zero-train optimization (ZTO) service that allows you to optimize a model's performance without additional training.

This will increase or decrease the effects of each training iteration on the model to obtain the best possible performance. In this case, the performance is measured by the semantic similarity between the model's response and the expected output.

The scaling factor limit is the maximum fraction of change that any one training iteration can undergo. In this case, each training iteration can only change by 10% of its original effect.

In [9]:
stimulus_paths = [f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)]
expected_output_path = "demo-models/Medical-Chatbot/stimuli/expected_output.txt"

response = client.dynamic.zero_train_optimizer(
    project_id=project["id"],
    scaling_factor_limit=0.1,
    stimulus_paths=stimulus_paths,
    batch_size=10,
    expected_output_path=expected_output_path,
    inference_config={"max_new_tokens": 100},
)

result = response["result"]

print(result)

{'model_path': '693325a950eb215a61201da7/results/e571dc44-88ef-4f25-be4a-dc9ff1708e5c.tar', 'trained_model_error': 0.9494965858757496, 'optimized_model_error': 0.9160460010170937, 'optimized_scaling_factors': [-0.04632406505941684, 0.07921454061469092, 0.07519154248678238, -0.09981340098743806], 'number_of_inferences': 45}


### Result details

`model_path` is the path to the optimized model in the Authentrics bucket.

In [10]:
result["model_path"]

'693325a950eb215a61201da7/results/e571dc44-88ef-4f25-be4a-dc9ff1708e5c.tar'

`base_model_error` is the semantic similarity between the trained model's response and the expected output.

In [11]:
result["trained_model_error"]

0.9494965858757496

`best_model_error` is the semantic similarity between the optimized model's response and the expected output.

In [12]:
result["optimized_model_error"]

0.9160460010170937

`optimized_scaling_factors` is the scaling factor for each training iteration that was applied to the model to produce the optimized model.

In [13]:
result["optimized_scaling_factors"]

[-0.04632406505941684,
 0.07921454061469092,
 0.07519154248678238,
 -0.09981340098743806]

`number_of_inferences` is the number of inferences that were performed to produce the optimized model.

In [14]:
result["number_of_inferences"]

45

### Inference the latest and optimized models

In [15]:
model_path = f"gs://authentrics-prod-bucket/{result['model_path']}"

inference(model_path, "Does my son have a cold? He has a fever of 39.2 and is vomiting.")

User prompt:

	Does my son have a cold? He has a fever of 39.2 and is vomiting.

Base model response:

	I can understand your concern.  If your child is experiencing a fever of 39.2 degrees Fahrenheit, vomiting, and other symptoms like cough, breathlessness, and chest pain, it is very likely that he has a respiratory infection. It is essential to monitor his condition closely and seek medical attention immediately if he experiences any of the following:  1. Difficulty breathing or wheezing.  2. Severe vomiting.  3. Fever that lasts more than three days.  4. Chest pain or difficulty breathing.  If you are concerned about your child's health, it is best to consult with a pediatrician. They can assess your child's condition and provide the appropriate treatment. In the meantime, you can follow the guidelines mentioned below:  1. Keep your child hydrated by offering plenty of fluids, such as water, clear broths, or electrolyte-rich beverages.  2. Offer warm liquids like tea, broth, or soup

### Analyze the optimized model

Upload the optimized model to the project


In [16]:
project = client.checkpoint.add_external_checkpoint(
    project_id,
    model_path.split("/", 3)[-1],
    authrx.FileType.HF_TEXT,
    file_name="optimized_model.tar",
    tag="optimized",
)
checkpoint_id = project["fileList"][-1]["id"]

Wait for analytics to be ready


In [17]:
analytics_ready = wait_for_analytics(project_id, min_wait_time=10, max_wait_time=300)

🔄 Waiting for analytics to process...


HTML(value='')

Run static analysis on the optimized model

In [18]:
response = client.static.static_analysis(
    project_id=project_id,
    checkpoint_id=checkpoint_id,
)

In [19]:
display_static_analysis_results(response["result"])


Weight summary score: 7.980081136338413e-05


,name,min,mean,std,max
0,base_model.model.model.layers.0.mlp.down_proj....,-1.850e-05,-1.197e-07,6.199e-06,1.725e-05
1,base_model.model.model.layers.0.mlp.down_proj....,-2.814e-05,-8.464e-07,9.854e-06,2.939e-05
2,base_model.model.model.layers.0.mlp.gate_proj....,-3.666e-05,2.842e-07,1.256e-05,4.831e-05
3,base_model.model.model.layers.0.mlp.gate_proj....,-1.795e-05,1.436e-07,4.928e-06,1.588e-05
4,base_model.model.model.layers.0.mlp.up_proj.lo...,-3.570e-05,-4.989e-07,1.217e-05,3.699e-05
5,base_model.model.model.layers.0.mlp.up_proj.lo...,-1.843e-05,3.024e-07,6.547e-06,1.562e-05
6,base_model.model.model.layers.0.self_attn.k_pr...,-3.873e-05,1.064e-07,1.386e-05,4.197e-05
7,base_model.model.model.layers.0.self_attn.k_pr...,-3.980e-05,8.078e-07,1.263e-05,5.255e-05
8,base_model.model.model.layers.0.self_attn.o_pr...,-3.551e-05,6.220e-07,1.191e-05,3.888e-05
9,base_model.model.model.layers.0.self_attn.o_pr...,-2.958e-05,4.794e-07,1.110e-05,3.026e-05
